In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os



import FMfuncs
import configs
import plot_func
# import FM2

import PGFMv2 as PGFM
# from torchdyn.core import NeuralODE
# from torchcfm.utils import torch_wrapper
# from torchcfm.utils import plot_trajectories
import time
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


In [ ]:
2

In [ ]:
FMclass = FMfuncs.OTFlowMatching()
PGFMclass = PGFM.PGFM()
# FM2class = FM2.FM2()

In [ ]:
if configs.d_model == 8:
    dataset = np.load(r'./data/MDMl2ball_dim8.npy')
elif configs.d_model == 20:
    dataset = np.load(r'./data/MDMl2ball_dim20.npy')
else:
    raise ValueError

dataset = torch.tensor(dataset,dtype=torch.float32, device=configs.device)
x_mat = PGFMclass.get_samples(dataset, 10000)
plot_func.plot_scatter_with_info(x_mat.cpu().numpy().transpose(), 0, 'unconstrained2',numItermax = 200000, distance = False)



In [ ]:
trained_model = FMclass.train(dataset, reflect=True)

In [ ]:
if configs.d_model == 8:
    ckpt_path = './saved_model/FM_l2ball_dim8_iter_1000000.pth'
elif configs.d_model == 20:
    ckpt_path = './saved_model/FM_l2ball_dim20_iter_1000000.pth'
else:
    raise ValueError

trained_model = PGFMclass.train2_2stage(dataset, ckpt_path)
# trained_model = FM2class.train2_2stage(dataset, ckpt_path)

In [ ]:
stage1_t_list = [0.0,0.2,0.4,0.6,0.8]
stage1_steps_list = [1, 15, 30, 45, 60]
stage2_steps_list = [75, 60, 45, 30, 15]
lambda_list = [2, 5, 10, 20, 30]
if configs.d_model == 8:
    ckpt_path = './saved_model/FM_l2ball_dim8_iter_1000000.pth'
elif configs.d_model == 20:
    ckpt_path = './saved_model/FM_l2ball_dim20_iter_1000000.pth'

for i in range(len(stage1_t_list)):
    for j in range(len(lambda_list)):
        stage1_t = stage1_t_list[i]
        stage2_steps = stage2_steps_list[i]
        stage1_steps = stage1_steps_list[i]
        lambda_val = lambda_list[j]
        PGFMclass = PGFM.PGFM(sig_min=0, stage1_t=stage1_t,
                         RL_Steps_S=stage2_steps, d_model=configs.d_model
                         , device=configs.device, default_generation_step= stage1_steps,
                         constraint_reward=lambda_val)
        save_name = 'PGFM_' + str(stage1_t).replace('.','p') +'_1s'+str(stage1_steps) + '_2s'+ str(stage2_steps) + '_lambda' + str(lambda_val)
        trained_model = PGFMclass.train2_2stage_batch(dataset, ckpt_path,
                                                    save_name, epoches=40000)


In [ ]:
stage1_t_list = [0.0,0.2,0.4,0.6,0.8]
stage1_steps_list = [1, 15, 30, 45, 60]
stage2_steps_list = [75, 60, 45, 30, 15]
lambda_list = [2, 5, 10, 20, 30]
if configs.d_model == 8:
    ckpt_path = './saved_model/FM_l2ball_dim8_iter_1000000.pth'
elif configs.d_model == 20:
    ckpt_path = './saved_model/FM_l2ball_dim20_iter_1000000.pth'

PGFMnum_out_mat = np.zeros([len(stage1_t_list), len(lambda_list)])
PGFMprob_mat = np.zeros([len(stage1_t_list), len(lambda_list)])
PGFMSWD_mat = np.zeros([len(stage1_t_list), len(lambda_list)])

PGFMnum_out_std_mat = np.zeros([len(stage1_t_list), len(lambda_list)])
PGFMprob_std_mat = np.zeros([len(stage1_t_list), len(lambda_list)])
PGFMSWD_std_mat = np.zeros([len(stage1_t_list), len(lambda_list)])

if configs.d_model == 8:
    ckpt1 = torch.load('./saved_model/FM_l2ball_dim8_iter_1000000.pth', map_location=configs.device, weights_only=True)
elif configs.d_model == 20:
    ckpt1 = torch.load('./saved_model/FM_l2ball_dim20_iter_1000000.pth', map_location=configs.device, weights_only=True)

stage1model = PGFMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)



for i in range(len(stage1_t_list)):
    for j in range(len(lambda_list)):
        stage1_t = stage1_t_list[i]
        stage2_steps = stage2_steps_list[i]
        stage1_steps = stage1_steps_list[i]
        lambda_val = lambda_list[j]
        PGFMclass = PGFM.PGFM(sig_min=0, stage1_t=stage1_t,
                         RL_Steps_S=stage2_steps, d_model=configs.d_model
                         , device=configs.device, default_generation_step= stage1_steps,
                         constraint_reward=lambda_val)
        save_name = 'PGFM_' + str(stage1_t).replace('.','p') +'_1s'+str(stage1_steps) + '_2s'+ str(stage2_steps) + '_lambda' + str(lambda_val)
        print(save_name)
        ckpt2 = torch.load('./saved_model_2/'+save_name+'.pth', map_location=configs.device, weights_only=True)
        stage2model = PGFMclass.policy
        stage2model.load_state_dict(ckpt2)

        PGFMnum_out_record = np.array([])
        PGFMprob_record = np.array([])
        PGFMSWD_record = np.array([])

        for k in range(100):
            reference = PGFMclass.get_samples(dataset, 10000).cpu().numpy()
            # resPGFM = PGFMclass.PGFMsample(stage1model, stage2model, 10000)
        
            resPGFM = PGFMclass.PGFMsample_train2(stage1model, stage2model, 10000)
            PGFMnum_out, PGFMprob, PGFMSWD = plot_func.only_info(resPGFM.cpu().numpy().transpose(), reference)
            
            PGFMnum_out_record = np.append(PGFMnum_out_record, PGFMnum_out)
            PGFMprob_record = np.append(PGFMprob_record, PGFMprob)
            PGFMSWD_record = np.append(PGFMSWD_record, PGFMSWD)

        PGFMnum_out_mat[i,j] = np.mean(PGFMnum_out_record)
        PGFMnum_out_std_mat[i,j] = np.std(PGFMnum_out_record)
        PGFMprob_mat[i,j] = np.mean(PGFMprob_record)
        PGFMprob_std_mat[i,j] = np.std(PGFMprob_record)
        PGFMSWD_mat[i,j] = np.mean(PGFMSWD_record)
        PGFMSWD_std_mat[i,j] = np.std(PGFMSWD_record)

In [ ]:
stage1_t_list = [0.0,0.2,0.4,0.6,0.8]
stage1_steps_list = [1, 15, 30, 45, 60]
stage2_steps_list = [75, 60, 45, 30, 15]
lambda_list = [2, 5, 10, 20, 30]
PGFMprob_mat = np.array([[0.014286, 0.007084, 0.004498, 0.003909, 0.001362],
       [0.010152, 0.007765, 0.005415, 0.002874, 0.002453],
       [0.011823, 0.006616, 0.004146, 0.002457, 0.002099],
       [0.013152, 0.008599, 0.00463 , 0.002513, 0.002408],
       [0.013447, 0.008579, 0.005559, 0.002513, 0.001926]])
PGFMSWD_mat = np.array([[0.00875843, 0.00980626, 0.01302159, 0.01279321, 0.01661936],
       [0.00920035, 0.01024121, 0.01225555, 0.01613421, 0.01548791],
       [0.00924858, 0.00983376, 0.01085441, 0.0143669 , 0.01424318],
       [0.00835777, 0.00936468, 0.01087226, 0.01205686, 0.01297447],
       [0.0090576 , 0.0098378 , 0.01095702, 0.01324996, 0.01324961]])

DDFMprob_mat = np.array([0.004158,0.002158,0.001365, 0.000767, 0.000675])
DDFMSWD_mat = np.array([0.0075,0.0080,0.0084, 0.0097, 0.0100])

In [ ]:
for i in range(PGFMprob_mat.shape[0]):
    plt.plot(PGFMSWD_mat[i], PGFMprob_mat[i], label = "FM-PG, $t_0=$"+str(stage1_t_list[i]), marker = '.')

plt.plot(DDFMSWD_mat, DDFMprob_mat, label = "FM-DD", marker = '.')

plt.scatter([0.0087], [0.0908], label = 'FM')
# plt.scatter([0.0086], [0.000502], label = 'FM-DD')
plt.legend()
plt.xlabel("SWD",fontsize=15)
plt.ylabel("$\\mathbb{P}(X_1\\notin C)$",fontsize=15)
plt.semilogy()
plt.grid()
# plt.title("$\\mathbb{P}(\\hat{x}_1\\notin C)$ vs. SWD for different $t_0$ by sweeping $\\lambda$")
plt.xticks(fontsize=15) 
plt.yticks(fontsize=15)
plt.savefig('./fig/d20compare.png',dpi=300 ,bbox_inches='tight')
plt.show()
    

In [ ]:
if configs.d_model == 8:
    ckpt_path = './saved_model/FM_l2ball_dim8_iter_1000000.pth'
elif configs.d_model == 20:
    ckpt_path = './saved_model/FM_l2ball_dim20_iter_100000.pth'
else:
    raise ValueError

trained_model = PGFMclass.train2_2stage_w_diff_distance(dataset, ckpt_path)

In [ ]:
if configs.d_model == 8:
    ckpt1 = torch.load('./saved_model/FM_l2ball_dim8_iter_1000000.pth', map_location=configs.device, weights_only=True)
    # ckpt1 = torch.load('./saved_model/RFM_l2ball_dim8_iter_200000.pth', map_location=configs.device, weights_only=True)
    # ckpt2 = torch.load('./saved_model/RFM_l2ball_dim8_iter_100000.pth', map_location=configs.device, weights_only=True)
    ckpt2 = torch.load('./saved_model/Apr17FM_l2balls2_dim8_iter_diffd_10000.pth', map_location=configs.device, weights_only=True)#FM2_l2balls2_dim8_iter_10000 FM2t0_l2balls2_dim8_iter_6000
elif configs.d_model == 20:
    ckpt1 = torch.load('./saved_model/FM_l2ball_dim20_iter_1000000.pth', map_location=configs.device, weights_only=True)
    # ckpt1 = torch.load('./saved_model/RFM_l2ball_dim20_iter_200000.pth', map_location=configs.device, weights_only=True)
    # ckpt2 = torch.load('./saved_model/DDFM_l2balls2_20_dim20_iter_diffd_5000.pth', map_location=configs.device, weights_only=True)#Apr17FM2_l2balls2_dim20_iter_40000
    ckpt2 = torch.load('./saved_model/Apr17FM_l2balls2_dim20_iter_diffd_10000.pth', map_location=configs.device, weights_only=True)
    # ckpt2 = torch.load('./saved_model/Apr8FM_l2balls2_dim20_iter_40000.pth', map_location=configs.device, weights_only=True)
    

else:
    raise ValueError

In [ ]:
# PGFMclass = FM2class

stage1model = PGFMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)

stage2model = PGFMclass.policy
# stage2model = PGFMclass.get_untrained_model()
stage2model.load_state_dict(ckpt2)

res = PGFMclass.PGFMsample_train2(stage1model, stage2model, 10000)


reference = PGFMclass.get_samples(dataset, 10000).cpu().numpy()
plot_func.plot_scatter_with_info(res.cpu().numpy().transpose(), reference, 'l2d8PGFM',numItermax = 200000, distance = True)



In [ ]:
stage1model = FMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)


res = FMfuncs.sampler(stage1model, 10000, stoptime=1, reflect=False)
reference = FMclass.get_samples(dataset, 10000).cpu().numpy()
plot_func.plot_scatter_with_info(res.cpu().numpy().transpose(), reference, 'l2d20FM',numItermax = 200000, distance = True)



In [ ]:
# PGFMclass = FM2class

FMnum_out_record = np.array([])
FMprob_record = np.array([])
FMSWD_record = np.array([])
PGFMnum_out_record = np.array([])
PGFMprob_record = np.array([])
PGFMSWD_record = np.array([])

for i in range(100):
    if i%10 == 0:
        print(i)
    reference = PGFMclass.get_samples(dataset, 10000).cpu().numpy()
    resFM = FMfuncs.sampler(stage1model, 10000, stoptime=1, reflect=False)
    # resPGFM = PGFMclass.PGFMsample(stage1model, stage2model, 10000)

    resPGFM = PGFMclass.PGFMsample_train2(stage1model, stage2model, 10000)
    FMnum_out, FMprob, FMSWD = plot_func.only_info(resFM.cpu().numpy().transpose(), reference)
    PGFMnum_out, PGFMprob, PGFMSWD = plot_func.only_info(resPGFM.cpu().numpy().transpose(), reference)
    
    FMnum_out_record = np.append(FMnum_out_record, FMnum_out)
    FMprob_record = np.append(FMprob_record, FMprob)
    FMSWD_record = np.append(FMSWD_record, FMSWD)
    PGFMnum_out_record = np.append(PGFMnum_out_record, PGFMnum_out)
    PGFMprob_record = np.append(PGFMprob_record, PGFMprob)
    PGFMSWD_record = np.append(PGFMSWD_record, PGFMSWD)
    





In [ ]:
print(" FM SWD: ", f"{np.mean( FMSWD_record):.4f}", r"\pm", f"{np.std( FMSWD_record):.4f}")
print(" FM num out: ", f"{np.mean( FMnum_out_record):.4f}", r"\pm", f"{np.std( FMnum_out_record):.4f}")

print("PGFM SWD: ", f"{np.mean(PGFMSWD_record):.4f}", r"\pm", f"{np.std(PGFMSWD_record):.4f}")
print("PGFM num out: ", f"{np.mean(PGFMnum_out_record):.4f}", r"\pm", f"{np.std(PGFMnum_out_record):.4f}")

In [ ]:
np.savez( './result_record/' +'train2_l2d20_record.npz', FMnum_out_record=FMnum_out_record, FMprob_record=FMprob_record,
        FMSWD_record=FMSWD_record,
        PGFMnum_out_record=PGFMnum_out_record,
        PGFMprob_record=PGFMprob_record,
        PGFMSWD_record=PGFMSWD_record)